In [24]:
import PyPDF2
import requests
import tabula
import pandas as pd
import json
import hashlib
import io

In [25]:
def get_hash_of_html(html_string):
    hash_object = hashlib.md5(html_string.encode('utf-8'))
    hash_of_html = hash_object.hexdigest()
    return hash_of_html

In [26]:
def get_text(bbox,reader,page_img):
    new = page_img.crop(bbox)
    bounds = reader.readtext(np.array(new), paragraph= True, x_ths = 2.0)
    lst = [bounds[i][1] for i in range(len(bounds))]
    return lst

In [27]:
def to_json(dictionary):
    hash_obj = json.dumps(dictionary)
    with open("dictionary.json", "w") as ts:
        json.dump(dictionary, ts,indent=4)
#         ts.write(hash_obj)

In [28]:
data_list = []
link = 'https://waynecountyne.gov/DocumentCenter/View/74/Active-Warrant-List?bidId='
user_agent = "scrapping_script/1.0"
headers = {'User-Agent': user_agent}
r = requests.get(link, headers=headers, stream = True)

In [33]:
def get_data(slug_name):
    file = io.BytesIO(r.content)
    fileReader = PyPDF2.PdfFileReader(file)

    pages = fileReader.numPages
    columns = ['LAST NAME', 'FIRST NAME', 'M.I.', 'CASE #']
    dictionary = {}
    for col in columns:
        dictionary[col] = []
    for page in range(1, pages+1):    
        try:
            df_ = tabula.read_pdf(link, pages = page)[0]
            df_ = df_.fillna('')
            print(len(df_))
            for row in range(len(df_)):
                if len(list(df_.iloc[row]))==5:
                    list_ = list(df_.iloc[row])
                    for i in range(len(columns)):
#                         print("i am here")
                        dictionary[columns[i]].append(list_[i])
        except:
            pass
    data_dict = {
                    "fullName" : "",
                    "firstName" : "",
                    "lastName" : "",
                    "additionalInfo" : "",
                    "summary" : ""
            }
    previousName = ""
    summary= ""
    additionalInfo = ""
    case = []
    info = []
    length = len(dictionary['LAST NAME'])
    for i in range(length):
        lastName = dictionary['LAST NAME'][i].strip()
        firstName = dictionary['FIRST NAME'][i].strip()
        fullName = firstName + " " + lastName
        print(fullName)
        info = [dictionary['M.I.'][i].strip()]
        case = [dictionary['CASE #'][i].strip()]
        if fullName != previousName and len(data_dict['fullName'])>0:
#             print(fullName)
#             print(previousName)
#             print(data_dict['fullName'])
            case = list(set(case))
            info = list(set(info))
            case = ", ".join(case)
            info = ", ".join(info)
            summary = data_dict['fullName'] + " has cases " + (case)
            additionalInfo = "M.I: " + (info) + ", Case #: " + (case)
            data_dict['summary'] = summary
            data_dict['additionalInfo'] = additionalInfo
            data_list.append(data_dict)
            data_dict = {
                    "fullName" : "",
                    "firstName" : "",
                    "lastName" : "",
                    "additionalInfo" : "",
                    "summary" : ""
            }
        if fullName and fullName == previousName:
            if case:
                case.append(case)
                print("here")
            if info:
                info.append(info)
        else:
            if fullName:
                data_dict['fullName'] = fullName
            if firstName:
                data_dict['firstName'] = firstName
            if lastName:
                data_dict['lastName'] = lastName
            if additionalInfo:
                data_dict['additionalInfo'] = additionalInfo
            if summary:
                data_dict['summary'] = summary
        previousName = fullName
    return data_list

In [34]:
if __name__ == "__main__":
    data_list = get_data('add-slug-here')
    to_json(data_list)

40
40
0
MICHAEL ADKINS-PENROD
MICHAEL ADKINS-PENROD
here
FERNANDO ALMANZA
NICHOLAS ALVAREZ
NOLASCO ANTONIO TOMAS
NOLASCO ANTONIO TOMAS
here
GLENDA ARNOLD
JULIO BARNETT GONZALEZ
DANIEL BERG
MICHAEL BLEXRUD
JEFF BRIDGES
URIAH BUFFALO CHIEF TORREZ II
JERAD BURWELL
HEATHER CALKINS
JULIAN CALLEJA
DIEGO CHAVEZ DE LA ISLA
PAMELA COX
PAMELA COX
here
RAESEAN CRATTON
ANETRA DECOSEY
JASE DENTON
FEDERICO DIEGO-SEBASTIAN
FEDERICO DIEGO-SEBASTIAN
here
RHONA DUFEK
RHONA DUFEK
here
VICTOR FERNANDO
CHRISTOPHER FLOWERS
ALEJANDRO GARCIA
ALEJANDRO GARCIA
here
ANTONIO GUERRA JR.
DOMINGO GUTIERREZ-JUAREZ
AUTUMN HANSEN
FERNANDO HERMOSILLO
JOSE HERMOSILLO JR.
OSCAR JIMENEZ
OSCAR JIMENEZ
here
ADONIAS JUAREZ OROZCO
TROY KOUNKEL
CALVIN LANDE
TODD LERCH
ALPACINO LEWIS
JUAN LOPEZ CIPRIAN
JUAN LOPEZ CIPRIAN
here
MARBIN LOPEZ-MORALES
LESLIE LUNDQUIST
TOMAS MARCOS
JOURNEE MARTIN
NEVIN MASQUAT
DAVID MATTHIES
FERNANDO MURILLO
JACKLYN NEWSAM
NEFTALI ORDONEZ-GARCIA
JUAN PAREDES RAMIREZ
DAMON PARKER
JUAN PEDRO RAYMUNDO
KE